In [129]:
import pyodbc 
import pandas as pd



In [130]:
#NOTES

##all new_events = to_insert_events

#new fights = fights_df

# new fight details  = new_fight_details

# fighters to update = fighters_df

#new fighters to inster = new_fighters_df


# Θα κάνω σύνδεση με sql θα πάρω τα τελευταία events και θα πάω να κάνω screip αν υπάρχει νέο που δεν είναι σε αυτά τα 10 το πέρνω

In [131]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

print("CONNECTED")

CONNECTED


In [132]:
last_events_query ='select top 10 * from events order by id desc' 
last_events = pd.read_sql(last_events_query,conn)


C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\2571800834.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  last_events = pd.read_sql(last_events_query,conn)


In [133]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

print('ok')

URL = "http://ufcstats.com/statistics/events/completed?page=all"

options = Options()

options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

driver.get(URL)

# περίμενε να φορτώσουν τα rows
WebDriverWait(driver, 20).until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "tr.b-statistics__table-row")
    )
)

html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

rows = soup.select("tr.b-statistics__table-row")

events = []

# ΙΔΙΑ λογική με requests version
for i, row in enumerate(rows):

    link = row.find("a")

    if link:

        event_name = link.text.strip()

        event_url = link["href"]

        cells = row.find_all("td")

        date = cells[0].text.strip()

        location = cells[1].text.strip()

        events.append({
            "event_name": event_name,
            "event_url": event_url,
            "date": date,
            "location": location
        })

    # ίδιο break logic
    if i > 10:
        break

driver.quit()

print(events)
print(len(events))

ok
[{'event_name': 'UFC Fight Night: Allen vs. Costa', 'event_url': 'http://ufcstats.com/event-details/73abb7a5c57fb443', 'date': 'UFC Fight Night: Allen vs. Costa\n                        \n\n                          May 16, 2026', 'location': 'Las Vegas, Nevada, USA'}, {'event_name': 'UFC 328: Chimaev vs. Strickland', 'event_url': 'http://ufcstats.com/event-details/9eedac48b497de5a', 'date': 'UFC 328: Chimaev vs. Strickland\n                        \n\n                          May 09, 2026', 'location': 'Newark, New Jersey, USA'}, {'event_name': 'UFC Fight Night: Della Maddalena vs. Prates', 'event_url': 'http://ufcstats.com/event-details/872b018076f831b0', 'date': 'UFC Fight Night: Della Maddalena vs. Prates\n                        \n\n                          May 02, 2026', 'location': 'Perth, Western Australia, Australia'}, {'event_name': 'UFC Fight Night: Sterling vs. Zalal', 'event_url': 'http://ufcstats.com/event-details/e60d773a0a42048a', 'date': 'UFC Fight Night: Sterling

In [134]:
# θα κάνω ένα καθάρισμα να τα φέρω σε σωστή μορφή 
#θα τα πάρω χωρίς id και αν δω κάποιο νέο θα πάω να το ρίξω με το νέο σωστό id 

df_events = pd.DataFrame(events)
df_events['date'] = df_events['date'].astype(str).str[-18:].str.strip()

month = []
day = []
year = []
for i in range(df_events.shape[0]):
    broken = df_events['date'][i].split(' ')
    month.append(broken[0])
    day.append(str(broken[1]).replace(',',''))
    year.append(broken[2])

df_events['day'] = day
df_events['month'] = month
df_events['year'] = year
df_events.drop(columns=['date'],inplace=True)

city=[]
region = []
country = []

for i in range(df_events.shape[0]):
    broken = df_events['location'][i].split(',')
    if len(broken) == 3:
        city.append(broken[0])
        region.append(broken[1])
        country.append(broken[2])


    if len(broken) <3 :
        city.append(broken[0])
        country.append(broken[1])
        region.append('unknown')
    
df_events['city'] = city
df_events['region'] = region
df_events['country'] = country

df_events.drop(columns=['location'],inplace=True)


card = []
for i in range(df_events.shape[0]):
    if 'UFC Fight Night' in df_events['event_name'][i]:
        card.append(0)
    else:
        card.append(1)
card

df_events['main_card'] = card
df_events = df_events[['event_name','main_card','year','country','month','day','city','region','event_url']]


In [137]:
one = df_events['event_name']
zero = last_events['event_name']
new = set(one)-set(zero)

if len(new) == 0 :
    print('Δεν υπαρχουν νέα events')
else :
    to_insert_events = df_events[df_events['event_name'].isin(new)]

Δεν υπαρχουν νέα events


## εδω αν δεν υπάρχουν νέα πρεέπει να σταματήσω 

In [ ]:
idd = int(last_events.iloc[0]['id'])
new_ids = [i+idd+1 for i in range(len(new))]

to_insert_events['ID'] = new_ids
to_insert_events = to_insert_events[['ID','event_name','main_card','year','country','month','day','city','region','event_url']]

In [21]:
to_insert_events

,ID,event_name,main_card,year,country,month,day,city,region,event_url
0,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...


# Είμαστε έτοιμοι με τον έλεγχο και ότι χρειάζεται για τα events τώρα πάμε στο επόμενο στάδιο που είναι ανα event να πάρουμε τα fight και τα fight_details 


In [22]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

# =====================================================
# DRIVER
# =====================================================

options = Options()

options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# =====================================================
# GET FIGHTS
# =====================================================

fight_links = to_insert_events['event_url'].tolist()

new_fights = []

for link in fight_links:

    print(f"Loading: {link}")

    driver.get(link)

    # περίμενε να φορτώσει το table
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "tr.b-fight-details__table-row")
        )
    )

    html = driver.page_source

    soup = BeautifulSoup(html, "html.parser")

    rows = soup.select("tr.b-fight-details__table-row")

    # ίδιο logic
    event_name = soup.find(
        "span",
        class_="b-content__title-highlight"
    ).text.strip()

    for row in rows:

        fight_url = row.get("data-link")

        cells = row.find_all("td")

        if len(cells) < 10:
            continue

        fighters = cells[1].find_all("a")

        fighter_1 = fighters[0].text.strip() if len(fighters) > 0 else None
        fighter_2 = fighters[1].text.strip() if len(fighters) > 1 else None

        # stats
        kd = cells[2].text.strip()

        strikes = cells[3].text.strip()

        td = cells[4].text.strip()

        sub = cells[5].text.strip()

        # other info
        weight_class = cells[6].text.strip()

        method = cells[7].text.strip()

        round_ended = cells[8].text.strip()

        time = cells[9].text.strip()

        new_fights.append({

            "event_name": event_name,

            "fight_url": fight_url,

            "fighter_1": fighter_1,
            "fighter_2": fighter_2,

            "kd": kd,
            "strikes": strikes,
            "td": td,
            "sub": sub,

            "weight_class": weight_class,

            "method": method,

            "round": round_ended,

            "time": time
        })

driver.quit()

print(new_fights[:3])
print(len(new_fights))

Loading: http://ufcstats.com/event-details/73abb7a5c57fb443
[{'event_name': 'UFC Fight Night: Allen vs. Costa', 'fight_url': 'http://ufcstats.com/fight-details/e4aa608124896794', 'fighter_1': 'Arnold Allen', 'fighter_2': 'Melquizael Costa', 'kd': '1\n          \n\n            \n            0', 'strikes': '98\n\n          \n\n\n            \n            100', 'td': '7\n          \n\n            \n            0', 'sub': '0\n          \n\n            \n            0', 'weight_class': 'Featherweight', 'method': 'U-DEC', 'round': '5', 'time': '5:00'}, {'event_name': 'UFC Fight Night: Allen vs. Costa', 'fight_url': 'http://ufcstats.com/fight-details/fc1266e2892ed111', 'fighter_1': 'Dooho Choi', 'fighter_2': 'Daniel Santos', 'kd': '1\n          \n\n            \n            0', 'strikes': '72\n\n          \n\n\n            \n            72', 'td': '0\n          \n\n            \n            0', 'sub': '0\n          \n\n            \n            0', 'weight_class': 'Featherweight', 'method': '

In [23]:
# πέιρα όλα τα fights που έγιναν στα events και τώρα πάω για καθάρισμα
# για τα kds 
fights_df = pd.DataFrame(new_fights)


fighter1_kd = []
fighter2_kd = []


kds = fights_df['kd']
for kd in kds:
    fighter1_kd.append(kd.replace('\n','')[:10].strip())
    fighter2_kd.append(kd.replace('\n','')[-10:].strip())


# για τα strikes 

fighter1_strikes = []
fighter2_strikes = []


strikes = fights_df['strikes']
for strike in strikes:
    fighter1_strikes.append(strike.replace('\n','')[:10].strip())
    fighter2_strikes.append(strike.replace('\n','')[-10:].strip())

# gia to td 


fighter1_td = []
fighter2_td = []


tds = fights_df['td']
for td in tds:
    fighter1_td.append(td.replace('\n','')[:10].strip())
    fighter2_td.append(td.replace('\n','')[-10:].strip())

# gia to sub 

fighter1_sub = []
fighter2_sub = []


subs = fights_df['sub']
for sub in subs:
    fighter1_sub.append(sub.replace('\n','')[:10].strip())
    fighter2_sub.append(sub.replace('\n','')[-10:].strip())


fights_df['fighter1_kd'] = fighter1_kd
fights_df['fighter2_kd'] = fighter2_kd

fights_df['fighter1_strikes'] = fighter1_strikes
fights_df['fighter2_strikes'] = fighter2_strikes

fights_df['fighter1_td'] = fighter1_td
fights_df['fighter2_td'] = fighter2_td

fights_df['fighter1_sub'] = fighter1_sub
fights_df['fighter2_sub'] = fighter2_sub


methods_d = fights_df['method']
methods = []
for method in methods_d:
    methods.append(method.replace('\n','').replace('              ',' ').strip())
fights_df['method'] = methods
fights_df.drop(columns=['kd','strikes','td','sub'],inplace=True)


# insert id 
id_for_fights = to_insert_events[['event_name','ID']]



fights_df = to_insert_events.merge(fights_df,on='event_name',how='left')

fights_df = fights_df.rename(columns={'ID': 'event_id'})




In [24]:
fights_df.head(2)

,event_id,event_name,main_card,year,country,month,day,city,region,event_url,...,round,time,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub
0,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...,...,5,5:00,1,0,98,100,7,0,0,0
1,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...,...,2,4:29,1,0,72,72,0,0,0,0


In [25]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

# =====================================================
# DRIVER
# =====================================================

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# =====================================================
# INPUT LINKS
# =====================================================

new_links = fights_df['fight_url'].tolist()

results = []

# =====================================================
# SCRAPE
# =====================================================

for i, url in enumerate(new_links):

    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".b-fight-details__person-name")
            )
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        names = soup.select(".b-fight-details__person-name")
        statuses = soup.select(".b-fight-details__person-status")

        if len(names) < 2 or len(statuses) < 2:
            continue

        fighter_1 = names[0].get_text(strip=True)
        fighter_2 = names[1].get_text(strip=True)

        status_1 = statuses[0].get_text(strip=True)
        status_2 = statuses[1].get_text(strip=True)

        if status_1 == "W":
            winner = fighter_1
        elif status_2 == "W":
            winner = fighter_2
        else:
            winner = None

        results.append({
            "fight_url": url,
            "winner": winner
        })

        print(f"{i+1}/{len(new_links)}")

        time.sleep(0.5)

    except Exception as e:
        print("error:", url)
        time.sleep(2)
        continue

driver.quit()

wins = pd.DataFrame(results, columns=['fight_url', 'winner'])

print(wins.head())

1/13
2/13
3/13
4/13
5/13
6/13
7/13
8/13
9/13
10/13
11/13
12/13
13/13
                                           fight_url              winner
0  http://ufcstats.com/fight-details/e4aa60812489...        Arnold Allen
1  http://ufcstats.com/fight-details/fc1266e2892e...          Dooho Choi
2  http://ufcstats.com/fight-details/ecb7ff543dd4...           Juan Diaz
3  http://ufcstats.com/fight-details/57bd683efc79...  Modestas Bukauskas
4  http://ufcstats.com/fight-details/a6ec8573a9d3...       Benardo Sopaj


In [26]:
fights_df = fights_df.merge(wins, on='fight_url',how='left')
fights_df['winner'] = fights_df['winner'].fillna('draw')


In [27]:
query_for_fight_ids = 'select top 1 fight_id from fights order by fight_id desc'


fight_ids = pd.read_sql(query_for_fight_ids,conn)
new_fight_ids = [i+int(fight_ids.loc[0].tolist()[0]) for i in range(len(fights_df['fight_url'].tolist()),0,-1)]
fights_df['fight_id'] = new_fight_ids


C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\1356297371.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fight_ids = pd.read_sql(query_for_fight_ids,conn)


In [28]:
import numpy as np

fights_df['fighter2_kd'] = fights_df['fighter2_kd'].replace('--', np.nan)
fights_df['fighter1_kd'] = fights_df['fighter1_kd'].replace('--', np.nan)
fights_df['fighter1_strikes'] = fights_df['fighter1_strikes'].replace('--', np.nan)
fights_df['fighter2_strikes'] = fights_df['fighter2_strikes'].replace('--', np.nan)
fights_df['fighter1_td'] = fights_df['fighter1_td'].replace('--', np.nan)
fights_df['fighter2_td'] = fights_df['fighter2_td'].replace('--', np.nan)
fights_df['fighter1_sub'] = fights_df['fighter1_sub'].replace('--', np.nan)
fights_df['fighter2_sub'] = fights_df['fighter2_sub'].replace('--', np.nan)

fights_df = fights_df[['fight_id','fight_url','fighter_1','fighter_2','winner','weight_class','method','round','time',
                       'event_name', 'event_id', 'fighter1_kd', 'fighter2_kd',
                       'fighter1_strikes', 'fighter2_strikes', 'fighter1_td', 'fighter2_td',
                       'fighter1_sub', 'fighter2_sub']]

In [29]:
fights_df.head(2)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round,time,event_name,event_id,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub
0,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,774,1,0,98,100,7,0,0,0
1,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,774,1,0,72,72,0,0,0,0


In [30]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

# =====================================================
# DRIVER
# =====================================================

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# =====================================================
# INPUT
# =====================================================

new_fight_links = fights_df['fight_url'].tolist()


# =====================================================
# HELPERS (same as yours)
# =====================================================

def split_two_values(text):
    parts = text.split()
    half = len(parts) // 2
    return " ".join(parts[:half]), " ".join(parts[half:])


def parse_fight(url):

    driver.get(url)

    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".b-fight-details__person-name")
        )
    )

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # fighters
    fighters = [
        x.get_text(strip=True)
        for x in soup.select(".b-fight-details__person-name")
    ]

    if len(fighters) < 2:
        print(f"[SKIP] No fighters found: {url}")
        return None

    fighter1, fighter2 = fighters[0], fighters[1]

    # find sig strikes row
    rows = soup.select("tbody tr")

    sig_cols = None

    for row in rows:
        cols = [td.get_text(" ", strip=True) for td in row.find_all("td")]

        if len(cols) == 9 and "of" in cols[1] and "%" in cols[2]:
            sig_cols = cols
            break

    if not sig_cols:
        print(f"[SKIP] No stats row found: {url}")
        return None

    # split stats (same logic)
    f1_sig_str, f2_sig_str = split_two_values(sig_cols[1])
    f1_sig_pct, f2_sig_pct = split_two_values(sig_cols[2])
    f1_head, f2_head = split_two_values(sig_cols[3])
    f1_body, f2_body = split_two_values(sig_cols[4])
    f1_leg, f2_leg = split_two_values(sig_cols[5])
    f1_distance, f2_distance = split_two_values(sig_cols[6])
    f1_clinch, f2_clinch = split_two_values(sig_cols[7])
    f1_ground, f2_ground = split_two_values(sig_cols[8])

    return {
        "url": url,
        "fighter1": fighter1,
        "fighter2": fighter2,

        "fighter1_sig_str": f1_sig_str,
        "fighter2_sig_str": f2_sig_str,

        "fighter1_sig_pct": f1_sig_pct,
        "fighter2_sig_pct": f2_sig_pct,

        "fighter1_head": f1_head,
        "fighter2_head": f2_head,

        "fighter1_body": f1_body,
        "fighter2_body": f2_body,

        "fighter1_leg": f1_leg,
        "fighter2_leg": f2_leg,

        "fighter1_distance": f1_distance,
        "fighter2_distance": f2_distance,

        "fighter1_clinch": f1_clinch,
        "fighter2_clinch": f2_clinch,

        "fighter1_ground": f1_ground,
        "fighter2_ground": f2_ground,
    }


# =====================================================
# LOOP
# =====================================================

results = []

for i, url in enumerate(new_fight_links, start=1):

    print(f"[{i}/{len(new_fight_links)}] Processing: {url}")

    try:
        data = parse_fight(url)

        if data:
            results.append(data)

    except Exception as e:
        print(f"[ERROR] {url} -> {e}")

    time.sleep(0.3)


driver.quit()

new_fight_details = pd.DataFrame(results)

print(new_fight_details.head())

[1/13] Processing: http://ufcstats.com/fight-details/e4aa608124896794
[2/13] Processing: http://ufcstats.com/fight-details/fc1266e2892ed111
[3/13] Processing: http://ufcstats.com/fight-details/ecb7ff543dd41bf8
[4/13] Processing: http://ufcstats.com/fight-details/57bd683efc797fc1
[5/13] Processing: http://ufcstats.com/fight-details/a6ec8573a9d38c51
[6/13] Processing: http://ufcstats.com/fight-details/ec32745308e2b055
[7/13] Processing: http://ufcstats.com/fight-details/6242cda790f2085d
[8/13] Processing: http://ufcstats.com/fight-details/c9e0aa88afb81dab
[9/13] Processing: http://ufcstats.com/fight-details/5f7111b982bcf6e3
[10/13] Processing: http://ufcstats.com/fight-details/48cb604345f11766
[11/13] Processing: http://ufcstats.com/fight-details/b98ead4eff6bf872
[12/13] Processing: http://ufcstats.com/fight-details/b482d4452e4d0eee
[13/13] Processing: http://ufcstats.com/fight-details/39417b7d07c2fd83
                                                 url            fighter1  \
0  http://

In [31]:
# τώρα πρέπει να δώσουμε fight_id στα fight details 

fight_url = fights_df['fight_url']
fight_id = fights_df['fight_id']

fight_url_id = {
    fight_url[i]: int(fight_id[i])
    for i in range(len(fight_id))
}


fight_id_ = []

for i in range(len(fight_url)):
    fight_id_.append(fight_url_id.get(fight_url[i]))

new_fight_details['fight_id'] = fight_id_

# τώρα μέχρι εδώ έχουμε πάρει τα events όπως πρέπει , έχουμε πάρει τα fights και η μόνο εκρεμότητα που έχουν και αυτά αλλά και τα
# fight details είναι fighter ids όπου πρώτα πρέπει να πάρουμε τους fighters  να κάνουμε ανανέωση τα stats τους , να προσθέσουμε νέους αν υπάρχουν και 
# μετά να γίνει όλο το υπόλοιπο

### ο τρόπος να το κάνουμε αυτό θα είναι λίγο διαφορετικός , θα πέρνουμε τα ονόματα από όλους που είχαν αγώνα , θα ελέγχουμε στην βάση αν υπάρχει ήδη το όνομα ,
### αν υπάρχει θα κάνουμε αναζήτηση ξανά και θα ανανεώνουμε τα stats αλλιώς αν είναι νέος μαχητής θα ψάξουμε να βρούμε τα stats του 

In [32]:
get_all_fighter_names_query = 'select name from fighters'

names_af_all_fighters = pd.read_sql(get_all_fighter_names_query,conn)
names_f = names_af_all_fighters['name'].tolist()

C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\3514268177.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  names_af_all_fighters = pd.read_sql(get_all_fighter_names_query,conn)


In [33]:
fighter1_names_from_new_fights = fights_df['fighter_1'].tolist()
fighter2_names_from_new_fights = fights_df['fighter_2'].tolist()

new_fighters = []
to_update_fighters = []

for i in range(len(fighter1_names_from_new_fights)):
    if fighter1_names_from_new_fights[i] not in names_f:
        new_fighters.append(fighter1_names_from_new_fights[i])
    else:
        to_update_fighters.append(fighter1_names_from_new_fights[i])

for i in range(len(fighter2_names_from_new_fights)):
    if fighter2_names_from_new_fights[i] not in names_f:
        new_fighters.append(fighter2_names_from_new_fights[i])
    else:
        to_update_fighters.append(fighter2_names_from_new_fights[i])

In [34]:
# άρα προς το παρον έχουμε τα ονόματα και το τί πρέπει να κάνουμε 
#ξεκινάμε με το update που είναι πιο έυκολο

In [35]:
update_ = ','.join(['?'] * len(to_update_fighters))

get_urls_from_players_to_update = f"""
SELECT url
FROM fighters
WHERE name IN ({update_})
"""

updated_urls_from_fighters = pd.read_sql(
    get_urls_from_players_to_update,
    conn,
    params=to_update_fighters
)

C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\3866197601.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  updated_urls_from_fighters = pd.read_sql(


In [36]:
updated_urls_from_fighters

,url
0,http://ufcstats.com/fighter-details/e93b04e308...
1,http://ufcstats.com/fighter-details/b6c37948cb...
2,http://ufcstats.com/fighter-details/e8fe9c15d3...
3,http://ufcstats.com/fighter-details/9673a497a9...
4,http://ufcstats.com/fighter-details/2558ae2e56...
5,http://ufcstats.com/fighter-details/3e8118c1ab...
6,http://ufcstats.com/fighter-details/fc0a4053eb...
7,http://ufcstats.com/fighter-details/9c442aaf14...
8,http://ufcstats.com/fighter-details/5f61780fe5...
9,http://ufcstats.com/fighter-details/64ad3e3b0e...


In [41]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

# =====================================================
# DRIVER
# =====================================================

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# =====================================================
# INPUT
# =====================================================

fighter_links = updated_urls_from_fighters['url'].tolist()
results = []

# =====================================================
# LOOP
# =====================================================

for i, url in enumerate(fighter_links):

    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".b-content__title-highlight")
            )
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # =====================================================
        # BASIC INFO (same logic)
        # =====================================================

        name_tag = soup.select_one(".b-content__title-highlight")
        name = name_tag.text.strip() if name_tag else None

        nickname_tag = soup.select_one(".b-content__Nickname")
        nickname = nickname_tag.text.strip() if nickname_tag else None

        record_tag = soup.select_one(".b-content__title-record")
        record_text = record_tag.text.strip() if record_tag else ""

        record = record_text.replace("Record:", "").strip()

        wins = None
        losses = None
        draws = None

        try:
            wins, losses, draws = record.split("-")
        except:
            pass

        # =====================================================
        # INFO BOX (same logic)
        # =====================================================

        info = {}

        rows = soup.select(".b-list__box-list-item")

        for row in rows:

            text = row.text.strip().replace("\n", "")

            if ":" in text:

                key, value = text.split(":", 1)

                info[key.strip()] = value.strip()

        # =====================================================
        # BUILD DICT (same structure)
        # =====================================================

        fighter_data = {
            "url": url,
            "name": name,
            "nickname": nickname,
            "record": record,
            "wins": wins,
            "losses": losses,
            "draws": draws,
        }

        fighter_data.update(info)

        results.append(fighter_data)

        print(f"{i+1}/{len(fighter_links)} DONE -> {name}")

        time.sleep(0.5)

    except Exception as e:
        print(f"\nERROR AT {url}")
        print(e)

driver.quit()

fighters_df = pd.DataFrame(results)

fighters_df[['wins', 'losses', 'draws']] = (
    fighters_df[['wins', 'losses', 'draws']]
    .apply(pd.to_numeric, errors='coerce')
)

print(fighters_df.head())

1/22 DONE -> Dooho Choi
2/22 DONE -> Benardo Sopaj
3/22 DONE -> Daniel Barez
4/22 DONE -> Polyana Viana
5/22 DONE -> Khaos Williams
6/22 DONE -> Tuco Tokkos
7/22 DONE -> Nicolle Caliari
8/22 DONE -> Shauna Bannon
9/22 DONE -> Andre Petroski
10/22 DONE -> Ivan Erslan
11/22 DONE -> Daniel Santos
12/22 DONE -> Alice Ardelean
13/22 DONE -> Ketlen Vieira
14/22 DONE -> Timmy Cuamba
15/22 DONE -> Jacqueline Cavalcanti
16/22 DONE -> Malcolm Wellmaker
17/22 DONE -> Modestas Bukauskas
18/22 DONE -> Arnold Allen
19/22 DONE -> Nikolay Veretennikov
20/22 DONE -> Luis Gurule
21/22 DONE -> Melquizael Costa
22/22 DONE -> Cody Brundage
                                                 url            name  \
0  http://ufcstats.com/fighter-details/e93b04e308...      Dooho Choi   
1  http://ufcstats.com/fighter-details/b6c37948cb...   Benardo Sopaj   
2  http://ufcstats.com/fighter-details/e8fe9c15d3...    Daniel Barez   
3  http://ufcstats.com/fighter-details/9673a497a9...   Polyana Viana   
4  http://ufc

In [43]:
fighters_df.head(1)

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.
0,http://ufcstats.com/fighter-details/e93b04e308...,Dooho Choi,The Korean Superboy,17-4-1,17,4,1.0,"5' 10""",145 lbs.,"70""",Orthodox,"Apr 10, 1991",5.00,55%,4.43,57%,1.26,53%,59%,0.5


In [44]:
# Τώρα θα πάρω το id τους και θα πάω να κάνω concat και να κάνω update σύφμωνα με αυτό 
update_ = ','.join(['?'] * len(to_update_fighters))

get_ids_from_players_to_update = f"""
SELECT id
FROM fighters
WHERE name IN ({update_})
"""

updated_ids_from_fighters = pd.read_sql(
    get_ids_from_players_to_update,
    conn,
    params=to_update_fighters
)

C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\532327243.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  updated_ids_from_fighters = pd.read_sql(


In [ ]:
fighters_df = fighters_df.merge(updated_ids_from_fighters, left_index=True,right_index=True)


,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,...,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,id
0,http://ufcstats.com/fighter-details/e93b04e308...,Dooho Choi,The Korean Superboy,17-4-1,17,4,1.0,"5' 10""",145 lbs.,"70""",...,"Apr 10, 1991",5.00,55%,4.43,57%,1.26,53%,59%,0.5,1979
1,http://ufcstats.com/fighter-details/b6c37948cb...,Benardo Sopaj,Lion King,13-3-0,13,3,0.0,"5' 6""",135 lbs.,"66""",...,"Sep 25, 2000",4.39,58%,4.69,52%,2.43,60%,76%,0.8,1996
2,http://ufcstats.com/fighter-details/e8fe9c15d3...,Daniel Barez,,17-8-0,17,8,0.0,"5' 6""",125 lbs.,"66""",...,"Dec 10, 1988",4.60,48%,6.93,50%,1.71,29%,80%,0.5,2016
3,http://ufcstats.com/fighter-details/9673a497a9...,Polyana Viana,Dama de Ferro,13-9-0,13,9,0.0,"5' 5""",115 lbs.,"67""",...,"Jun 14, 1992",2.82,40%,2.74,52%,0.67,33%,33%,1.5,2044
4,http://ufcstats.com/fighter-details/2558ae2e56...,Khaos Williams,The OxFighter,16-5-0,16,5,0.0,"6' 0""",170 lbs.,"77""",...,"Mar 30, 1994",4.97,39%,5.34,41%,0.00,0%,51%,0.0,2075
5,http://ufcstats.com/fighter-details/3e8118c1ab...,Tuco Tokkos,,11-6-0,11,6,0.0,"6' 4""",205 lbs.,"76""",...,"Jun 30, 1990",2.59,41%,3.70,46%,2.87,36%,33%,0.6,2097
6,http://ufcstats.com/fighter-details/fc0a4053eb...,Nicolle Caliari,,9-4-0,9,4,0.0,"5' 3""",115 lbs.,"62""",...,"Nov 04, 1996",2.88,39%,5.59,52%,3.38,29%,50%,1.0,2100
7,http://ufcstats.com/fighter-details/9c442aaf14...,Shauna Bannon,Mama B,7-3-0,7,3,0.0,"5' 5""",115 lbs.,"65""",...,"Oct 23, 1993",3.85,44%,3.25,45%,0.00,0%,47%,0.5,2146
8,http://ufcstats.com/fighter-details/5f61780fe5...,Andre Petroski,,13-6-0,13,6,0.0,"6' 0""",185 lbs.,"73""",...,"Jun 12, 1991",2.75,49%,2.99,51%,3.13,52%,86%,1.2,2166
9,http://ufcstats.com/fighter-details/64ad3e3b0e...,Ivan Erslan,,15-6-0 (1 NC),15,6,NaN,"6' 2""",205 lbs.,"72""",...,"Nov 15, 1991",2.53,49%,4.57,54%,0.93,37%,70%,0.0,2172


In [ ]:
### τώρα θα πάμε να κάνουμε update στην βάση τα δεδομένα αυτών 

Index(['url', 'name', 'nickname', 'record', 'wins', 'losses', 'draws',
       'Height', 'Weight', 'Reach', 'STANCE', 'DOB', 'SLpM', 'Str. Acc.',
       'SApM', 'Str. Def', 'TD Avg.', 'TD Acc.', 'TD Def.', 'Sub. Avg.', 'id'],
      dtype='str')

In [48]:

cursor = conn.cursor()

query = """
UPDATE fighters
SET
    url = ?,
    name = ?,
    nickname = ?,
    record = ?,
    wins = ?,
    losses = ?,
    draws = ?
WHERE ID = ?
"""

data = []

for _, row in fighters_df.iterrows():

    data.append((
        row["url"],
        row["name"],
        row["nickname"],
        row["record"],
        int(row["wins"]) if pd.notnull(row["wins"]) else None,
        int(row["losses"]) if pd.notnull(row["losses"]) else None,
        int(row["draws"]) if pd.notnull(row["draws"]) else None,
        int(row["id"])
    ))

cursor.executemany(query, data)
conn.commit()

print("UPDATE DONE")

UPDATE DONE


In [49]:
# οκ μάλλον το κρατάμε ανοιχτό για δέυτερο μελλοντικό έλεγχο 

## τώρα πρέπει να πάμε να προσθέσουμε τους νέους μαχητές 

In [ ]:
new_fighters 

['Juan Diaz', 'Tommy Gantt', 'Christian Edwards', 'Artur Minev']

In [51]:
# για αυτους δεν έχουμε url επομένως μάλλον θα πρέπει να κάνουμε ένα screip ξανά τα events για να πάρουμε τα links , ίσως υπάρχει και κσλυτερος τρόπος αλλα
#προς το παρόν θα προχωρήσουμε έτσι καθώς είναι και θα είναι πάντα λίγοι επομένως δεν μας πειράζει

In [55]:
u = fights_df['fight_url'].tolist()
o = fights_df['fighter_1'].tolist()
t = fights_df['fighter_2'].tolist()
urls_to_check = []
for  i in range(len(fights_df)):
    if o[i] in new_fighters or t[i] in new_fighters:
        urls_to_check.append(u[i])


In [56]:
urls_to_check

['http://ufcstats.com/fight-details/ecb7ff543dd41bf8',
 'http://ufcstats.com/fight-details/57bd683efc797fc1',
 'http://ufcstats.com/fight-details/c9e0aa88afb81dab']

In [62]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC





# =========================
# DRIVER SETUP
# =========================
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)


# =========================
# RESULT STORAGE
# =========================
filtered_fighter_urls = set()


# =========================
# SCRAPING LOOP
# =========================
for url in urls_to_check:
    try:
        driver.get(url)

        # περιμένει να φορτώσει το page
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        time.sleep(1)

        # παίρνουμε όλα τα links προς fighters
        fighter_links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="fighter-details"]')

        for a in fighter_links:
            try:
                name = a.text.strip()
                href = a.get_attribute("href")

                if name and href:
                    if name in new_fighters:
                        filtered_fighter_urls.add(href)

            except Exception:
                continue

    except Exception as e:
        print(f"Error in {url}: {e}")
        continue


driver.quit()


# =========================
# OUTPUT
# =========================
filtered_fighter_urls = list(filtered_fighter_urls)

print("FOUND FIGHTERS:")
for f in filtered_fighter_urls:
    print(f)

FOUND FIGHTERS:
http://ufcstats.com/fighter-details/de83f920fd302871
http://ufcstats.com/fighter-details/20d37922ba85e6fc
http://ufcstats.com/fighter-details/f5d82eccdea4e2dd
http://ufcstats.com/fighter-details/74939a16c7b56f33


In [ ]:
filtered_fighter_urls # τα url απο τους νέους μαχητες τώρα θα πάω να τους κάνω screip τα απαρέτητα 

['http://ufcstats.com/fighter-details/de83f920fd302871',
 'http://ufcstats.com/fighter-details/20d37922ba85e6fc',
 'http://ufcstats.com/fighter-details/f5d82eccdea4e2dd',
 'http://ufcstats.com/fighter-details/74939a16c7b56f33']

In [64]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

# =====================================================
# DRIVER
# =====================================================

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# =====================================================
# INPUT
# =====================================================

fighter_links = filtered_fighter_urls
results = []

# =====================================================
# LOOP
# =====================================================

for i, url in enumerate(fighter_links):

    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".b-content__title-highlight")
            )
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # =====================================================
        # BASIC INFO (same logic)
        # =====================================================

        name_tag = soup.select_one(".b-content__title-highlight")
        name = name_tag.text.strip() if name_tag else None

        nickname_tag = soup.select_one(".b-content__Nickname")
        nickname = nickname_tag.text.strip() if nickname_tag else None

        record_tag = soup.select_one(".b-content__title-record")
        record_text = record_tag.text.strip() if record_tag else ""

        record = record_text.replace("Record:", "").strip()

        wins = None
        losses = None
        draws = None

        try:
            wins, losses, draws = record.split("-")
        except:
            pass

        # =====================================================
        # INFO BOX (same logic)
        # =====================================================

        info = {}

        rows = soup.select(".b-list__box-list-item")

        for row in rows:

            text = row.text.strip().replace("\n", "")

            if ":" in text:

                key, value = text.split(":", 1)

                info[key.strip()] = value.strip()

        # =====================================================
        # BUILD DICT (same structure)
        # =====================================================

        fighter_data = {
            "url": url,
            "name": name,
            "nickname": nickname,
            "record": record,
            "wins": wins,
            "losses": losses,
            "draws": draws,
        }

        fighter_data.update(info)

        results.append(fighter_data)

        print(f"{i+1}/{len(fighter_links)} DONE -> {name}")

        time.sleep(0.5)

    except Exception as e:
        print(f"\nERROR AT {url}")
        print(e)

driver.quit()

new_fighters_df = pd.DataFrame(results)

new_fighters_df[['wins', 'losses', 'draws']] = (
    new_fighters_df[['wins', 'losses', 'draws']]
    .apply(pd.to_numeric, errors='coerce')
)



1/4 DONE -> Tommy Gantt
2/4 DONE -> Artur Minev
3/4 DONE -> Christian Edwards
4/4 DONE -> Juan Diaz


In [65]:
new_fighters_df

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.
0,http://ufcstats.com/fighter-details/de83f920fd...,Tommy Gantt,,12-0-0 (1 NC),12,0,NaN,"5' 11""",155 lbs.,"76""",Orthodox,"Jan 09, 1993",2.92,75%,0.85,43%,11.29,36%,100%,1.4
1,http://ufcstats.com/fighter-details/20d37922ba...,Artur Minev,Headhunter,7-1-0,7,1,0.0,"5' 9""",155 lbs.,"69""",Orthodox,"Sep 27, 2003",1.02,61%,3.57,26%,0.00,0%,64%,0.0
2,http://ufcstats.com/fighter-details/f5d82eccde...,Christian Edwards,Pain,8-5-0,8,5,0.0,"6' 5""",205 lbs.,"78""",Southpaw,"Nov 05, 1998",2.40,50%,2.40,55%,0.00,0%,0%,0.0
3,http://ufcstats.com/fighter-details/74939a16c7...,Juan Diaz,Pegajoso,16-1-1,16,1,1.0,"5' 8""",135 lbs.,"69""",Orthodox,"Jun 03, 1998",3.72,41%,3.40,59%,5.50,50%,0%,0.8


In [74]:
#πρεπει να πάρουν ένα id άρα θα τρέξω ένα sql query klp...

get_last_id ='select top 1 id   from fighters order by id desc' 
last_id = pd.read_sql(get_last_id,conn)
last_id = last_id['id'].tolist()[0]


C:\Users\mplan\AppData\Local\Temp\ipykernel_14472\4075503207.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  last_id = pd.read_sql(get_last_id,conn)


In [85]:
ids_for_new_fighters = [i + last_id   for i in range(len(new_fighters_df['Height'].tolist()),0,-1)]

In [86]:
ids_for_new_fighters

[2676, 2675, 2674, 2673]

In [93]:
new_fighters_df.drop(columns=['id'],inplace=True)
new_fighters_df['ID'] = ids_for_new_fighters


In [94]:
new_fighters_df

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,...,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID
0,http://ufcstats.com/fighter-details/de83f920fd...,Tommy Gantt,,12-0-0 (1 NC),12,0,NaN,"5' 11""",155 lbs.,"76""",...,"Jan 09, 1993",2.92,75%,0.85,43%,11.29,36%,100%,1.4,2676
1,http://ufcstats.com/fighter-details/20d37922ba...,Artur Minev,Headhunter,7-1-0,7,1,0.0,"5' 9""",155 lbs.,"69""",...,"Sep 27, 2003",1.02,61%,3.57,26%,0.00,0%,64%,0.0,2675
2,http://ufcstats.com/fighter-details/f5d82eccde...,Christian Edwards,Pain,8-5-0,8,5,0.0,"6' 5""",205 lbs.,"78""",...,"Nov 05, 1998",2.40,50%,2.40,55%,0.00,0%,0%,0.0,2674
3,http://ufcstats.com/fighter-details/74939a16c7...,Juan Diaz,Pegajoso,16-1-1,16,1,1.0,"5' 8""",135 lbs.,"69""",...,"Jun 03, 1998",3.72,41%,3.40,59%,5.50,50%,0%,0.8,2673


In [98]:
#είναι έτοιμοι και οι νέοι μαχητές μένει να τους κάνουμε insert στην βάση μας 
new_fighters_df['draws'] = new_fighters_df['draws'].fillna(0)

In [99]:
import pyodbc


cursor = conn.cursor()

query = """
INSERT INTO fighters (
    url, name, nickname, record,
    wins, losses, draws,
    Height, Weight, Reach,
    STANCE, DOB,
    SLpM, [Str. Acc.], SApM,
    [Str. Def], [TD Avg.], [TD Acc.],
    [TD Def.], [Sub. Avg.], ID
)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

data = [
    (
        row["url"],
        row["name"],
        row["nickname"],
        row["record"],
        row["wins"],
        row["losses"],
        row["draws"],
        row["Height"],
        row["Weight"],
        row["Reach"],
        row["STANCE"],
        row["DOB"],
        row["SLpM"],
        row["Str. Acc."],
        row["SApM"],
        row["Str. Def"],
        row["TD Avg."],
        row["TD Acc."],
        row["TD Def."],
        row["Sub. Avg."],
        row["ID"]
    )
    for _, row in new_fighters_df.iterrows()
]

cursor.executemany(query, data)
conn.commit()

cursor.close()
conn.close()

print("DONE")

DONE


In [106]:
## ΑΨΟΓΑ ΜΕΧΡΙ ΕΔΩ ΜΕΝΕΙ ΜΟΝΟ ΝΑ ΔΩΣΟΥΜΕ FIGHTER ID ΣΕ FIGHTS KAI FIGHT_DETAILS ΚΑΙ ΕΙΜΑΣΤΕ ΟΚ !

# για να το κάνουμε αυτό θα πάρουμε όλους συννολικά τους μαχητές που έπαιξαν σε αυτές τις μαχες

fighters_df.head(2)

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,...,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,id
0,http://ufcstats.com/fighter-details/e93b04e308...,Dooho Choi,The Korean Superboy,17-4-1,17,4,1.0,"5' 10""",145 lbs.,"70""",...,"Apr 10, 1991",5.00,55%,4.43,57%,1.26,53%,59%,0.5,1979
1,http://ufcstats.com/fighter-details/b6c37948cb...,Benardo Sopaj,Lion King,13-3-0,13,3,0.0,"5' 6""",135 lbs.,"66""",...,"Sep 25, 2000",4.39,58%,4.69,52%,2.43,60%,76%,0.8,1996


In [107]:
fighters_df['ID'] = fighters_df['id']
fighters_df.drop(columns=['id'],inplace=True)
all_fg  = pd.concat([fighters_df, new_fighters_df], axis=0, ignore_index=True)
all_fg.shape

(26, 21)

In [108]:
all_fg.tail(5)

,url,name,nickname,record,wins,losses,draws,Height,Weight,Reach,...,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,ID
21,http://ufcstats.com/fighter-details/f14f644d41...,Cody Brundage,,12-9-1 (1 NC),12,9,NaN,"6' 0""",185 lbs.,"72""",...,"May 16, 1994",2.50,51%,2.95,50%,1.80,41%,66%,0.5,2430
22,http://ufcstats.com/fighter-details/de83f920fd...,Tommy Gantt,,12-0-0 (1 NC),12,0,0.0,"5' 11""",155 lbs.,"76""",...,"Jan 09, 1993",2.92,75%,0.85,43%,11.29,36%,100%,1.4,2676
23,http://ufcstats.com/fighter-details/20d37922ba...,Artur Minev,Headhunter,7-1-0,7,1,0.0,"5' 9""",155 lbs.,"69""",...,"Sep 27, 2003",1.02,61%,3.57,26%,0.00,0%,64%,0.0,2675
24,http://ufcstats.com/fighter-details/f5d82eccde...,Christian Edwards,Pain,8-5-0,8,5,0.0,"6' 5""",205 lbs.,"78""",...,"Nov 05, 1998",2.40,50%,2.40,55%,0.00,0%,0%,0.0,2674
25,http://ufcstats.com/fighter-details/74939a16c7...,Juan Diaz,Pegajoso,16-1-1,16,1,1.0,"5' 8""",135 lbs.,"69""",...,"Jun 03, 1998",3.72,41%,3.40,59%,5.50,50%,0%,0.8,2673


In [100]:
fights_df

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round,time,event_name,event_id,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub
0,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,774,1,0,98,100,7,0,0,0
1,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,774,1,0,72,72,0,0,0,0
2,8699,http://ufcstats.com/fight-details/ecb7ff543dd4...,Juan Diaz,Malcolm Wellmaker,Juan Diaz,Bantamweight,SUB Rear Naked Choke,2,4:08,UFC Fight Night: Allen vs. Costa,774,0,0,13,15,5,0,1,0
3,8698,http://ufcstats.com/fight-details/57bd683efc79...,Modestas Bukauskas,Christian Edwards,Modestas Bukauskas,Catch Weight,S-DEC,3,5:00,UFC Fight Night: Allen vs. Costa,774,0,0,36,36,0,0,0,0
4,8697,http://ufcstats.com/fight-details/a6ec8573a9d3...,Benardo Sopaj,Timmy Cuamba,Benardo Sopaj,Bantamweight,SUB Rear Naked Choke,2,2:25,UFC Fight Night: Allen vs. Costa,774,1,0,37,29,1,0,2,0
5,8696,http://ufcstats.com/fight-details/ec32745308e2...,Khaos Williams,Nikolay Veretennikov,Khaos Williams,Welterweight,KO/TKO Punches,1,3:31,UFC Fight Night: Allen vs. Costa,774,1,0,21,13,0,0,0,0
6,8695,http://ufcstats.com/fight-details/6242cda790f2...,Ivan Erslan,Tuco Tokkos,Ivan Erslan,Light Heavyweight,U-DEC,3,5:00,UFC Fight Night: Allen vs. Costa,774,0,0,37,54,2,0,0,1
7,8694,http://ufcstats.com/fight-details/c9e0aa88afb8...,Tommy Gantt,Artur Minev,Tommy Gantt,Lightweight,KO/TKO Punches,2,2:51,UFC Fight Night: Allen vs. Costa,774,0,0,28,8,6,0,0,0
8,8693,http://ufcstats.com/fight-details/5f7111b982bc...,Ketlen Vieira,Jacqueline Cavalcanti,Ketlen Vieira,Women's Bantamweight,U-DEC,3,5:00,UFC Fight Night: Allen vs. Costa,774,0,0,30,53,1,0,0,0
9,8692,http://ufcstats.com/fight-details/48cb604345f1...,Cody Brundage,Andre Petroski,Cody Brundage,Middleweight,KO/TKO Punches,2,0:44,UFC Fight Night: Allen vs. Costa,774,1,0,43,8,0,1,0,0


In [109]:
fighters1 = fights_df['fighter_1']
fighters2 = fights_df['fighter_2']

fighters_id = all_fg['ID']
fighters_names = all_fg['name']

fighter_name_id = {
    fighters_names[i]: int(fighters_id[i])
    for i in range(len(fighters_id))
}

fighter1_id = []
fighter2_id = []

for i in range(len(fighters1)):
    fighter1_id.append(fighter_name_id.get(fighters1[i]))
    fighter2_id.append(fighter_name_id.get(fighters2[i]))

In [111]:
fighter1_id = pd.Series(fighter1_id).astype('Int64')
fighter2_id = pd.Series(fighter2_id).astype('Int64')


fights_df['fighter1_id'] = fighter1_id
fights_df['fighter2_id'] = fighter2_id

In [113]:
fights_df.head(2)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round,time,event_name,...,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id
0,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,...,1,0,98,100,7,0,0,0,2322,2393
1,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,...,1,0,72,72,0,0,0,0,1979,2183


In [114]:
## ΤΟ ιδιο και σε fight _details 

In [116]:
new_fight_details.head(1)

,url,fighter1,fighter2,fighter1_sig_str,fighter2_sig_str,fighter1_sig_pct,fighter2_sig_pct,fighter1_head,fighter2_head,fighter1_body,fighter2_body,fighter1_leg,fighter2_leg,fighter1_distance,fighter2_distance,fighter1_clinch,fighter2_clinch,fighter1_ground,fighter2_ground,fight_id
0,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,98 of 152,100 of 254,64%,39%,81 of 134,47 of 173,6 of 7,20 of 42,11 of 11,33 of 39,79 of 131,97 of 249,0 of 2,0 of 1,19 of 19,3 of 4,8701


In [117]:
fighters1 = new_fight_details['fighter1']
fighters2 = new_fight_details['fighter2']

fighters_id = all_fg['ID']
fighters_names = all_fg['name']

fighter_name_id = {
    fighters_names[i]: int(fighters_id[i])
    for i in range(len(fighters_id))
}

fighter1_id = []
fighter2_id = []

for i in range(len(fighters1)):
    fighter1_id.append(fighter_name_id.get(fighters1[i]))
    fighter2_id.append(fighter_name_id.get(fighters2[i]))

In [ ]:
fighter1_id = pd.Series(fighter1_id).astype('Int64')
fighter2_id = pd.Series(fighter2_id).astype('Int64')


new_fight_details['fighter1_id'] = fighter1_id
new_fight_details['fighter2_id'] = fighter2_id

In [119]:
new_fight_details.head(2)


,url,fighter1,fighter2,fighter1_sig_str,fighter2_sig_str,fighter1_sig_pct,fighter2_sig_pct,fighter1_head,fighter2_head,fighter1_body,...,fighter2_leg,fighter1_distance,fighter2_distance,fighter1_clinch,fighter2_clinch,fighter1_ground,fighter2_ground,fight_id,fighter1_id,fighter2_id
0,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,98 of 152,100 of 254,64%,39%,81 of 134,47 of 173,6 of 7,...,33 of 39,79 of 131,97 of 249,0 of 2,0 of 1,19 of 19,3 of 4,8701,2322,2393
1,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,72 of 138,72 of 182,52%,39%,56 of 122,54 of 156,13 of 13,...,3 of 4,67 of 133,64 of 172,3 of 3,8 of 10,2 of 2,0 of 0,8700,1979,2183


## Μάλλον είναι όλα σχεδόν έτοιμα ελέγχω τα δεδομένα και τα ρίχνω στην βάση

#NOTES

##all new_events = to_insert_events

#new fights = fights_df

# new fight details  = new_fight_details

# fighters to update = fighters_df

#new fighters to inster = new_fighters_df


In [ ]:
to_insert_events.head(2) # ok 

,ID,event_name,main_card,year,country,month,day,city,region,event_url
0,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...


In [ ]:
new_fight_details.head(2) # ok 

,url,fighter1,fighter2,fighter1_sig_str,fighter2_sig_str,fighter1_sig_pct,fighter2_sig_pct,fighter1_head,fighter2_head,fighter1_body,...,fighter2_leg,fighter1_distance,fighter2_distance,fighter1_clinch,fighter2_clinch,fighter1_ground,fighter2_ground,fight_id,fighter1_id,fighter2_id
0,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,98 of 152,100 of 254,64%,39%,81 of 134,47 of 173,6 of 7,...,33 of 39,79 of 131,97 of 249,0 of 2,0 of 1,19 of 19,3 of 4,8701,2322,2393
1,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,72 of 138,72 of 182,52%,39%,56 of 122,54 of 156,13 of 13,...,3 of 4,67 of 133,64 of 172,3 of 3,8 of 10,2 of 2,0 of 0,8700,1979,2183


In [122]:
fights_df.head(2)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round,time,event_name,...,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub,fighter1_id,fighter2_id
0,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,...,1,0,98,100,7,0,0,0,2322,2393
1,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,...,1,0,72,72,0,0,0,0,1979,2183


In [127]:
new_fight_details.columns

Index(['url', 'fighter1', 'fighter2', 'fighter1_sig_str', 'fighter2_sig_str',
       'fighter1_sig_pct', 'fighter2_sig_pct', 'fighter1_head',
       'fighter2_head', 'fighter1_body', 'fighter2_body', 'fighter1_leg',
       'fighter2_leg', 'fighter1_distance', 'fighter2_distance',
       'fighter1_clinch', 'fighter2_clinch', 'fighter1_ground',
       'fighter2_ground', 'fight_id', 'fighter1_id', 'fighter2_id'],
      dtype='str')

In [124]:
# ola ok pame na kanoyme ta insert 


conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

cursor = conn.cursor()

query = """
INSERT INTO events (
    id,
    event_name,
    main_card,
    event_year,
    country,
    event_month,
    event_day,
    city,
    region,
    event_url
)
VALUES (?,?,?,?,?,?,?,?,?,?)
"""

data = [
    (
        row["ID"],
        row["event_name"],
        row["main_card"],
        row["year"],
        row["country"],
        row["month"],
        row["day"],
        row["city"],
        row["region"],
        row["event_url"]
    )
    for _, row in to_insert_events.iterrows()
]

cursor.executemany(query, data)
conn.commit()

cursor.close()
conn.close()

print("EVENTS INSERT DONE")

EVENTS INSERT DONE


In [126]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)


cursor = conn.cursor()

query = """
INSERT INTO fights (
    fight_id,
    fight_url,
    fighter_1,
    fighter_2,
    winner,
    weight_class,
    method,
    round_num,
    fight_time,
    event_name,
    event_id,
    fighter1_kd,
    fighter2_kd,
    fighter1_strikes,
    fighter2_strikes,
    fighter1_td,
    fighter2_td,
    fighter1_sub,
    fighter2_sub,
    fighter1_id,
    fighter2_id
)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

data = [
    (
        row["fight_id"],
        row["fight_url"],
        row["fighter_1"],
        row["fighter_2"],
        row["winner"],
        row["weight_class"],
        row["method"],
        row["round"],
        row["time"],
        row["event_name"],
        row["event_id"],
        row["fighter1_kd"],
        row["fighter2_kd"],
        row["fighter1_strikes"],
        row["fighter2_strikes"],
        row["fighter1_td"],
        row["fighter2_td"],
        row["fighter1_sub"],
        row["fighter2_sub"],
        row["fighter1_id"],
        row["fighter2_id"]
    )
    for _, row in fights_df.iterrows()
]

cursor.executemany(query, data)
conn.commit()

cursor.close()
conn.close()

print("FIGHTS INSERT DONE")

FIGHTS INSERT DONE


In [128]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

cursor = conn.cursor()

query = """
INSERT INTO fight_details (
    url,
    fighter1,
    fighter2,
    fighter1_sig_str,
    fighter2_sig_str,
    fighter1_sig_pct,
    fighter2_sig_pct,
    fighter1_head,
    fighter2_head,
    fighter1_body,
    fighter2_body,
    fighter1_leg,
    fighter2_leg,
    fighter1_distance,
    fighter2_distance,
    fighter1_clinch,
    fighter2_clinch,
    fighter1_ground,
    fighter2_ground,
    fight_id,
    fighter1_id,
    fighter2_id
)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

data = [
    (
        row["url"],
        row["fighter1"],
        row["fighter2"],
        row["fighter1_sig_str"],
        row["fighter2_sig_str"],
        row["fighter1_sig_pct"],
        row["fighter2_sig_pct"],
        row["fighter1_head"],
        row["fighter2_head"],
        row["fighter1_body"],
        row["fighter2_body"],
        row["fighter1_leg"],
        row["fighter2_leg"],
        row["fighter1_distance"],
        row["fighter2_distance"],
        row["fighter1_clinch"],
        row["fighter2_clinch"],
        row["fighter1_ground"],
        row["fighter2_ground"],
        row["fight_id"],
        row["fighter1_id"],
        row["fighter2_id"]
    )
    for _, row in new_fight_details.iterrows()
]

cursor.executemany(query, data)
conn.commit()

cursor.close()
conn.close()

print("FIGHT_DETAILS INSERT DONE")

FIGHT_DETAILS INSERT DONE


# 100% !!!! 